# Meal Type Classifier Notebook

This notebook downloads the Food.com dataset from Kaggle, filters recipes with clear meal-type tags (breakfast, lunch, dinner, snack), trains a text classifier, evaluates it, and exports a reusable model.

In [1]:
# Setup: install dependencies (run once per environment)
import sys, subprocess, pkgutil
def pip_install(pkgs):
    for p in pkgs:
        if pkgutil.find_loader(p.split('==')[0]) is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])

pip_install([
    'kagglehub',
    'pandas',
    'numpy',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'joblib'
])
print('Dependencies installed.')

/var/folders/t4/8mgg99sj739dlpkvq4dvspwc0000gn/T/ipykernel_27600/2302706486.py:5: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(p.split('==')[0]) is None:


  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
Using cached idna-3.10-py3-none-any.whl (70 kB)



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 45.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 44.4 MB/s eta 0:00:00
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 42.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 49.5 MB/s eta 0:00:0000:0100:01



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 22.0 MB/s eta 0:00:00a 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 35.4 MB/s eta 0:00:00
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


Dependencies installed.



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Download the Food.com dataset via kagglehub
import kagglehub
data_path = kagglehub.dataset_download('shuyangli94/food-com-recipes-and-user-interactions')
print('Path to dataset files:', data_path)
import os, glob
print('Files:', glob.glob(os.path.join(data_path, '**', '*'), recursive=True)[:20])

/Users/gabrieltouchette/Documents/IDontCook/api/api/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 267M/267M [00:06<00:00, 43.7MB/s] 

Extracting files...


Path to dataset files: /Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2
Files: ['/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/interactions_train.csv', '/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/interactions_test.csv', '/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/ingr_map.pkl', '/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/interactions_validation.csv', '/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/PP_recipes.csv', '/Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_interactions.csv', '/Users/gabrieltouchette/.cache/kagglehub/dat

In [3]:
# Load relevant CSVs (recipes and tags)
import os
import pandas as pd
recipes_csv = None
interactions_csv = None
# Common names in this dataset
candidates = ['RAW_recipes.csv', 'RAW_interactions.csv', 'recipes.csv', 'interactions.csv']
for c in candidates:
    p = os.path.join(data_path, c)
    if os.path.exists(p):
        if 'recipe' in c.lower():
            recipes_csv = p
        elif 'interaction' in c.lower():
            interactions_csv = p

assert recipes_csv is not None, 'Recipes CSV not found in dataset folder'
print('Using recipes file:', recipes_csv)
df = pd.read_csv(recipes_csv)
print(df.shape)
df.head()

Using recipes file: /Users/gabrieltouchette/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_recipes.csv
(231637, 12)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


## Filter recipes with clear meal-type tags
We'll infer meal types from the tags and/or recipe names. We'll keep only those clearly labeled as one of breakfast, lunch, dinner, snack.

In [11]:
import re
import numpy as np
import ast


def parse_tags(tags_val):
    # Robustly parse tags from a stringified list or comma-separated string
    if isinstance(tags_val, list):
        return [str(t).lower().strip() for t in tags_val]
    if isinstance(tags_val, str):
        s = tags_val.strip()
        try:
            if s.startswith('[') and s.endswith(']'):
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(t).lower().strip() for t in parsed]
        except Exception:
            pass
        # Fallback: split on commas/semicolons
        parts = [p.strip().lower() for p in re.split(r'[;,]', s) if p.strip()]
        return parts
    return []

BREAKFAST = {'breakfast', 'brunch', 'pancake', 'waffle', 'oatmeal'}
LUNCH = {'lunch', 'sandwich', 'wrap', 'salad'}
DINNER = {'dinner', 'entree', 'main-dish', 'main dish', 'supper'}
SNACK = {'snack', 'appetizer', 'finger-food', 'finger food'}
DESSERT = {'dessert', 'cake', 'cookie', 'pie', 'sweet', 'chocolate', 'ice cream', 'candy', 'brownie', 'pudding', 'tart', 'cheesecake', 'cupcake', 'frosting', 'icing', 'fudge'}


def infer_meal_type(title, tags):
    text = str(title or '').lower()
    tagset = set(tags or [])
    # Priority order to avoid overlaps - dessert gets highest priority as it's most specific
    if tagset & DESSERT or re.search(r"\b(dessert|cake|cookie|pie|sweet|chocolate|brownie|pudding|tart|cheesecake|cupcake|fudge|candy|ice cream)s?\b", text):
        return 'dessert'
    if tagset & BREAKFAST or re.search(r"\b(breakfast|brunch|pancake|oatmeal|waffle)s?\b", text):
        return 'breakfast'
    if tagset & LUNCH or re.search(r"\b(lunch|sandwich|wrap|salad)s?\b", text):
        return 'lunch'
    if tagset & DINNER or re.search(r"\b(dinner|entree|main dish|supper)s?\b", text):
        return 'dinner'
    if tagset & SNACK or re.search(r"\b(snack|appetizer|finger food)s?\b", text):
        return 'snack'
    return None


df['tags_parsed'] = df.get('tags', np.nan).apply(parse_tags)
df['meal_type'] = df.apply(lambda r: infer_meal_type(r.get('name'), r.get('tags_parsed', [])), axis=1)
labeled = df.dropna(subset=['meal_type']).copy()
print('Labeled recipes:', labeled.shape)
print('Meal type distribution:')
print(labeled['meal_type'].value_counts())
labeled[['name', 'meal_type']].head(10)

Labeled recipes: (155248, 14)
Meal type distribution:
meal_type
dinner       57735
dessert      46552
lunch        26581
breakfast    22384
snack         1996
Name: count, dtype: int64


,name,meal_type
1,a bit different breakfast pizza,breakfast
2,all in the kitchen chili,dinner
3,alouette potatoes,breakfast
7,backyard style barbecued ribs,dessert
8,bananas 4 ice cream pie,dessert
9,beat this banana bread,breakfast
10,berry good sandwich spread,lunch
15,chicken lickin good pork chops,dinner
17,chinese candy,dessert
18,chinese chop suey,dinner


## Prepare text features and train/test split
We'll use the recipe title and optionally the description/ingredients if present.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

def build_text(row):
    parts = []
    for key in ['name', 'description', 'ingredients', 'steps', 'tags_parsed']:
        v = row.get(key) if isinstance(row, dict) else row[key] if key in row else None
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if isinstance(v, list):
            parts.extend([str(x) for x in v])
        else:
            parts.append(str(v))
    return ' '.join(parts)

labeled['text'] = labeled.apply(build_text, axis=1)
X = labeled['text'].fillna('')
y = labeled['meal_type']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', X_train.shape, 'Test size:', X_test.shape)

Train size: (124198,) Test size: (31050,)


## Train a text classifier
We'll start with a linear model (Logistic Regression) over character+word TF-IDF.

In [13]:
from sklearn.metrics import accuracy_score
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.9)),
    ('clf', LogisticRegression(max_iter=200, class_weight='balanced', n_jobs=None, verbose=0)),
])
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred, labels=['breakfast','lunch','dinner','snack','dessert'])
print('Confusion Matrix:')
print(cm)

Accuracy: 0.942415458937198
              precision    recall  f1-score   support

   breakfast       0.91      0.91      0.91      4477
     dessert       0.97      0.90      0.94      9311
      dinner       0.95      0.98      0.97     11547
       lunch       0.93      0.95      0.94      5316
       snack       0.64      0.93      0.76       399

    accuracy                           0.94     31050
   macro avg       0.88      0.94      0.90     31050
weighted avg       0.94      0.94      0.94     31050

Confusion Matrix:
[[ 4082   119   129    44   103]
 [   32  5046    91    78    69]
 [   41    39 11368    33    66]
 [    4     5     6   371    13]
 [  340   203   319    54  8395]]


## Export the trained model
We'll save the pipeline with joblib for reuse in the Django backend.

In [14]:
import joblib, os
out_dir = '../api/recipes/models'
os.makedirs(out_dir, exist_ok=True)
model_path = os.path.abspath(os.path.join(out_dir, 'meal_type_tfidf_logreg.joblib'))
joblib.dump(pipeline, model_path)
print('Saved model to:', model_path)

Saved model to: /Users/gabrieltouchette/Documents/IDontCook/api/recipes/models/meal_type_tfidf_logreg.joblib


## Inference helper
A convenience function to classify new titles/recipes.

In [15]:
def predict_meal_type(texts):
    if isinstance(texts, str):
        texts = [texts]
    return pipeline.predict(texts)

predict_meal_type(['Blueberry Pancakes', 'Turkey Sandwich', 'Sheet-Pan Chicken Dinner', 'Energy Bars', 'Chocolate Chip Cookies'])

array(['breakfast', 'lunch', 'dinner', 'lunch', 'dessert'], dtype=object)